# laya RLCD training on Kaggle (parameterized)

Runtime deps with zero GitHub DNS dependency:
- laya 0.4.6 from PyPI (the exact version validated locally)
- splits from the jsdelivr GitHub mirror (Kaggle's DNS
  cannot resolve github.com; the CDN mirror is reachable)
- multilingual checkpoint from HF Hub, revision-pinned
  (a9597e0 -- the checkpoint the gold was distilled from)

Headless via the push->poll->pull pattern (see docs/training-plan.md 6.1):

```bash
kaggle kernels push -p cloud/kaggle/   # upload triggers a GPU run
kaggle kernels status hathawayandrx105/train    # poll until complete
kaggle kernels output hathawayandrx105/train -p out/
```

The checkpoint is pushed to HF Hub (set the HF_TOKEN secret in
Settings -> Secrets); the report lands in /kaggle/working
(pulled by `kernels output`).

In [ ]:
EPOCHS = 4
UNFREEZE = False
HF_REPO = "hathawayandrx105/laya-canon-head"

In [ ]:
import pathlib
import subprocess
import urllib.request

subprocess.run(["pip", "install", "laya==0.4.6"], check=True)

# splits via the jsdelivr GitHub mirror
base = ("https://cdn.jsdelivr.net/gh/"
        "hathawayandrx105/laya-canon-judgments@main/"
        "data/splits/")
out = pathlib.Path("/kaggle/working/splits")
out.mkdir(exist_ok=True)
for name in ("train.jsonl", "validation.jsonl", "test.jsonl",
             "manifest.json"):
    urllib.request.urlretrieve(base + name, out / name)
    print("fetched", name)

In [ ]:
from huggingface_hub import snapshot_download

from laya.train import TrainConfig, finetune
from laya.agent import _fix_tokenizer_config

# Pin the base revision: a bare repo id resolves to whatever
# main is today (two runs trained different base models --
# choice accuracy 6/6 vs 2/6 on the same calibration slice).
# a9597e0 is the multilingual checkpoint the gold
# distributions were distilled from.
snap = snapshot_download(
    "convaiinnovations/laya",
    revision="a9597e0e438d6971d2eb6742d7d8610cefd9f262",
    local_dir="/kaggle/working/laya-snap",
    allow_patterns=["multilingual*"],
)
model_dir = snap + "/multilingual"
_fix_tokenizer_config(model_dir)
config = TrainConfig(
    epochs=EPOCHS, micro_batch=4, grad_accum=2,
    freeze_encoder=not UNFREEZE, seed=42, calib_seed=42,
    max_len=1024, head_max_len=256,
)
summary = finetune(
    data="/kaggle/working/splits/train.jsonl",
    model_dir=model_dir,
    output_dir="/kaggle/working/out",
    config=config,
    device="cuda",
)
print(summary)

In [ ]:
import json
import os

from huggingface_hub import HfApi

with open("/kaggle/working/report.json", "w") as f:
    json.dump({"summary": str(summary), "epochs": EPOCHS,
               "unfreeze": UNFREEZE}, f, indent=2)

api = HfApi(token=os.environ.get("HF_TOKEN"))
if api:
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, folder_path="/kaggle/working/out",
                      path_in_repo="")
    print("checkpoint pushed to", HF_REPO)
else:
    print("HF_TOKEN not set; checkpoint stays in /kaggle/working/out")